# Hair Segmentation Notebook

This notebook provides tools to segment hair from facial images and apply different hair colors.

## Install Required Packages

In [ ]:
!pip install torch torchvision numpy opencv-python pillow

## Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from google.colab import files  # For Google Colab
import requests
import io

## Define Model Architecture

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_chan, out_chan, ks=3, stride=1, padding=1):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(in_chan, out_chan, kernel_size=ks, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_chan)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class BiSeNetOutput(nn.Module):
    def __init__(self, in_chan, mid_chan, n_classes):
        super(BiSeNetOutput, self).__init__()
        self.conv = ConvBNReLU(in_chan, mid_chan, ks=3, stride=1, padding=1)
        self.conv_out = nn.Conv2d(mid_chan, n_classes, kernel_size=1, bias=False)

    def forward(self, x):
        x = self.conv(x)
        x = self.conv_out(x)
        return x

class AttentionRefinementModule(nn.Module):
    def __init__(self, in_chan, out_chan):
        super(AttentionRefinementModule, self).__init__()
        self.conv = ConvBNReLU(in_chan, out_chan, ks=3, stride=1, padding=1)
        self.conv_atten = nn.Conv2d(out_chan, out_chan, kernel_size=1, bias=False)
        self.bn_atten = nn.BatchNorm2d(out_chan)
        self.sigmoid_atten = nn.Sigmoid()

    def forward(self, x):
        feat = self.conv(x)
        atten = torch.mean(feat, dim=(2, 3), keepdim=True)
        atten = self.conv_atten(atten)
        atten = self.bn_atten(atten)
        atten = self.sigmoid_atten(atten)
        out = torch.mul(feat, atten)
        return out

class ContextPath(nn.Module):
    def __init__(self, backbone='resnet18'):
        super(ContextPath, self).__init__()
        self.backbone_name = backbone
        if backbone == 'resnet18':
            self.backbone = resnet18(pretrained=True)
            self.arm16 = AttentionRefinementModule(256, 128)
            self.arm32 = AttentionRefinementModule(512, 128)
            self.conv_head32 = ConvBNReLU(128, 128, ks=3, stride=1, padding=1)
            self.conv_head16 = ConvBNReLU(128, 128, ks=3, stride=1, padding=1)
            self.conv_avg = ConvBNReLU(512, 128, ks=1, stride=1, padding=0)

    def forward(self, x):
        feat8, feat16, feat32 = self.backbone(x)
        
        avg = torch.mean(feat32, dim=(2, 3), keepdim=True)
        avg = self.conv_avg(avg)
        
        feat32_arm = self.arm32(feat32)
        feat32_sum = feat32_arm + avg
        feat32_up = nn.functional.interpolate(feat32_sum, size=feat16.size()[2:], mode='nearest')
        feat32_up = self.conv_head32(feat32_up)

        feat16_arm = self.arm16(feat16)
        feat16_sum = feat16_arm + feat32_up
        feat16_up = nn.functional.interpolate(feat16_sum, size=feat8.size()[2:], mode='nearest')
        feat16_up = self.conv_head16(feat16_up)

        return feat8, feat16_up, feat32_up

class FeatureFusionModule(nn.Module):
    def __init__(self, in_chan, out_chan):
        super(FeatureFusionModule, self).__init__()
        self.convblk = ConvBNReLU(in_chan, out_chan, ks=1, stride=1, padding=0)
        self.conv1 = nn.Conv2d(out_chan, out_chan//4, kernel_size=1, stride=1, padding=0, bias=False)
        self.conv2 = nn.Conv2d(out_chan//4, out_chan, kernel_size=1, stride=1, padding=0, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, fsp, fcp):
        fcat = torch.cat([fsp, fcp], dim=1)
        feat = self.convblk(fcat)
        atten = torch.mean(feat, dim=(2, 3), keepdim=True)
        atten = self.conv1(atten)
        atten = self.relu(atten)
        atten = self.conv2(atten)
        atten = self.sigmoid(atten)
        feat_atten = torch.mul(feat, atten)
        feat_out = feat_atten + feat
        return feat_out

class BiSeNet(nn.Module):
    def __init__(self, n_classes=19, backbone='resnet18'):
        super(BiSeNet, self).__init__()
        self.cp = ContextPath(backbone)
        self.ffm = FeatureFusionModule(256 + 128, 256)
        self.conv_out = BiSeNetOutput(256, 256, n_classes)
        self.conv_out16 = BiSeNetOutput(128, 64, n_classes)
        self.conv_out32 = BiSeNetOutput(128, 64, n_classes)

    def forward(self, x):
        H, W = x.size()[2:]
        feat_res8, feat_cp8, feat_cp16 = self.cp(x)
        feat_fuse = self.ffm(feat_res8, feat_cp8)

        feat_out = self.conv_out(feat_fuse)
        feat_out16 = self.conv_out16(feat_cp8)
        feat_out32 = self.conv_out32(feat_cp16)

        feat_out = nn.functional.interpolate(feat_out, size=(H, W), mode='bilinear', align_corners=True)
        feat_out16 = nn.functional.interpolate(feat_out16, size=(H, W), mode='bilinear', align_corners=True)
        feat_out32 = nn.functional.interpolate(feat_out32, size=(H, W), mode='bilinear', align_corners=True)
        return feat_out, feat_out16, feat_out32

def resnet18(pretrained=True):
    model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=pretrained)
    
    features = []
    for name, module in model.named_children():
        if name == 'conv1':
            features.append(module)
        elif name == 'bn1':
            features.append(module)
        elif name == 'relu':
            features.append(module)
        elif name == 'maxpool':
            features.append(module)
        elif name == 'layer1':
            features.append(module)
        elif name == 'layer2':
            features.append(module)
        elif name == 'layer3':
            features.append(module)
        elif name == 'layer4':
            features.append(module)
    
    backbone = torch.nn.Sequential(*features)
    
    def forward(x):
        feat4 = backbone[:6](x)  # layer1
        feat8 = backbone[6](feat4)  # layer2
        feat16 = backbone[7](feat8)  # layer3
        feat32 = backbone[8](feat16)  # layer4
        return feat8, feat16, feat32
    
    backbone.forward = forward
    return backbone

## Download Pre-trained Model

In [ ]:
# Create directory for the model
os.makedirs('pretrained', exist_ok=True)

# Download the model file
# Note: You need to manually download the model from https://drive.google.com/open?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
# and upload it to the notebook or save it as 'pretrained/79999_iter.pth'

# For Google Colab, you can use this to upload the model file
try:
    from google.colab import files
    print("Please upload the pre-trained model file (79999_iter.pth)")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, 'pretrained/79999_iter.pth')
    print("Model file saved as 'pretrained/79999_iter.pth'")
except ImportError:
    print("Not running in Google Colab. Please ensure the model file is in 'pretrained/79999_iter.pth'")

## Define Hair Segmentation Functions

In [ ]:
def extract_hair_mask(image, model_path):
    """
    Extract hair mask from an image
    
    Args:
        image: PIL Image or path to image
        model_path: path to the pretrained model
    
    Returns:
        hair_mask: binary mask of hair region
    """
    # Load model
    n_classes = 19
    net = BiSeNet(n_classes=n_classes)
    net.load_state_dict(torch.load(model_path, map_location='cpu'))
    net.eval()
    
    # Use GPU if available
    if torch.cuda.is_available():
        net = net.cuda()
    
    # Load and preprocess image
    if isinstance(image, str):
        img = Image.open(image)
    else:
        img = image
        
    to_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    
    # Resize to 512x512
    img_resized = img.resize((512, 512), Image.BILINEAR)
    img_tensor = to_tensor(img_resized)
    img_tensor = torch.unsqueeze(img_tensor, 0)
    
    if torch.cuda.is_available():
        img_tensor = img_tensor.cuda()
    
    # Forward pass
    with torch.no_grad():
        out = net(img_tensor)[0]
    
    # Hair class index is 17 in CelebAMask-HQ
    hair_idx = 17
    parsing = out.squeeze(0).cpu().numpy().argmax(0)
    
    # Create hair mask
    hair_mask = np.zeros_like(parsing)
    hair_mask[parsing == hair_idx] = 255
    
    # Resize mask to original image size
    hair_mask = cv2.resize(hair_mask, (img.width, img.height), interpolation=cv2.INTER_NEAREST)
    
    return hair_mask

def apply_hair_color(image, hair_mask, color=(0, 0, 255)):
    """
    Apply color to hair region
    
    Args:
        image: PIL Image or path to image
        hair_mask: binary mask of hair region
        color: RGB color tuple to apply to hair (default: red)
    
    Returns:
        colored_img: image with colored hair
    """
    # Load image
    if isinstance(image, str):
        img = cv2.imread(image)
    else:
        img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    
    # Create color overlay
    color_overlay = np.zeros_like(img)
    color_overlay[:] = color[::-1]  # BGR format for OpenCV
    
    # Apply color to hair region
    mask_3d = cv2.cvtColor(hair_mask, cv2.COLOR_GRAY2BGR) / 255.0
    colored_img = img * (1 - mask_3d * 0.7) + color_overlay * mask_3d * 0.7
    colored_img = colored_img.astype(np.uint8)
    
    return colored_img

def show_results(original_img, hair_mask, colored_img):
    """
    Display the original image, hair mask, and colored image
    
    Args:
        original_img: original image (PIL Image or numpy array)
        hair_mask: hair mask (numpy array)
        colored_img: image with colored hair (numpy array)
    """
    plt.figure(figsize=(15, 5))
    
    # Convert PIL Image to numpy array if needed
    if isinstance(original_img, Image.Image):
        original_img = np.array(original_img)
    
    # Convert BGR to RGB for display
    if colored_img.shape[-1] == 3:  # Check if it's a color image
        colored_img = cv2.cvtColor(colored_img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(hair_mask, cmap='gray')
    plt.title('Hair Mask')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(colored_img)
    plt.title('Colored Hair')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

## Upload an Image

In [ ]:
# For Google Colab, you can use this to upload an image
try:
    from google.colab import files
    print("Please upload an image")
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
    img = Image.open(image_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
except ImportError:
    # If not in Colab, specify a local image path
    image_path = "your_image.jpg"  # Change this to your image path
    img = Image.open(image_path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## Extract Hair Mask and Apply Color

In [ ]:
# Path to the pre-trained model
model_path = 'pretrained/79999_iter.pth'

# Extract hair mask
hair_mask = extract_hair_mask(img, model_path)

# Define color (R, G, B)
color = (255, 0, 0)  # Red

# Apply color to hair
colored_img = apply_hair_color(img, hair_mask, color)

# Show results
show_results(img, hair_mask, colored_img)

## Try Different Colors

In [ ]:
# Define different colors to try
colors = {
    'Red': (255, 0, 0),
    'Green': (0, 255, 0),
    'Blue': (0, 0, 255),
    'Yellow': (255, 255, 0),
    'Purple': (255, 0, 255),
    'Cyan': (0, 255, 255),
    'Black': (0, 0, 0),
    'White': (255, 255, 255),
    'Brown': (165, 42, 42),
    'Pink': (255, 192, 203),
    'Orange': (255, 165, 0)
}

# Create a grid of colored hair images
fig = plt.figure(figsize=(15, 10))
rows = 3
cols = 4

# Add original image
ax = fig.add_subplot(rows, cols, 1)
ax.imshow(img)
ax.set_title('Original')
ax.axis('off')

# Add colored versions
i = 2
for color_name, color_value in colors.items():
    if i <= rows * cols:
        colored = apply_hair_color(img, hair_mask, color_value)
        colored_rgb = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
        
        ax = fig.add_subplot(rows, cols, i)
        ax.imshow(colored_rgb)
        ax.set_title(color_name)
        ax.axis('off')
        i += 1

plt.tight_layout()
plt.show()

## Custom Color Picker

In [ ]:
# For interactive color selection (works in Jupyter Notebook)
from ipywidgets import interact, widgets

@interact(r=(0, 255, 1), g=(0, 255, 1), b=(0, 255, 1))
def update_color(r=255, g=0, b=0):
    color = (r, g, b)
    colored_img = apply_hair_color(img, hair_mask, color)
    colored_rgb = cv2.cvtColor(colored_img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(colored_rgb)
    plt.title(f'Hair Color: RGB({r}, {g}, {b})')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

## Save the Results

In [ ]:
# Save the hair mask and colored image
cv2.imwrite('hair_mask.png', hair_mask)
cv2.imwrite('colored_hair.png', colored_img)

# For Google Colab, you can download the results
try:
    from google.colab import files
    files.download('hair_mask.png')
    files.download('colored_hair.png')
except ImportError:
    print("Files saved as 'hair_mask.png' and 'colored_hair.png'")